# 04 - Fine-tuning (LoRA/PEFT) do assistente medico

**Este notebook foi desenhado para rodar no Google Colab (GPU gratuita, ex.: T4).**
Localmente (Windows, sem GPU) o treino seria lento demais; o codigo abaixo e correto e testado no fluxo de dados, mas a celula de treino em si deve ser executada no Colab.

Passos:
1. Clonar o repositorio no Colab (ou fazer upload de `data/medical_corpus/`).
2. Instalar as dependencias de fine-tuning.
3. Carregar o modelo base + aplicar LoRA.
4. Treinar sobre `train.jsonl`, avaliar sobre `eval.jsonl`.
5. Salvar o adapter em `results/finetuning/lora_adapter/` e baixar de volta para o repo local (usado por `src/assistant/llm_backend.py`).

In [11]:
# No Colab, descomente e rode:
# !rm -rf stroke-prediction # Adicionado para garantir uma clonagem limpa
# !git clone https://github.com/BrunaNicolau/stroke-prediction stroke-prediction
# %cd stroke-prediction

import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT = Path('/content/drive/MyDrive/stroke-prediction')
    sys.path.insert(0, str(ROOT))
    print("Rodando no Google Colab")
else:
    ROOT = Path('..').resolve()
    print("Rodando Localmente")

!pip install -r "{ROOT / 'requirements.txt'}"
# !pip install -q bitsandbytes  # opcional, acelera em GPU

Mounted at /content/drive
Rodando no Google Colab


In [12]:
import os

from src.finetuning.dataset import build_hf_dataset, load_split

train_examples = load_split(os.path.join(ROOT, 'data', 'medical_corpus', 'train.jsonl'))
eval_examples = load_split(os.path.join(ROOT, 'data', 'medical_corpus', 'eval.jsonl'))
print(f'{len(train_examples)} exemplos de treino, {len(eval_examples)} de avaliacao')

train_dataset = build_hf_dataset(train_examples)
train_dataset[0]

510 exemplos de treino, 89 de avaliacao


{'text': '### Instrução:\nWho is at risk for Carotid Artery Disease? ?\n\n### Resposta:\nThe major risk factors for carotid artery disease, listed below, also are the major risk factors for coronary heart disease (also called coronary artery disease) and peripheral artery disease. Diabetes. With this disease, the bodys blood sugar level is too high because the body doesnt make enough insulin or doesnt use its insulin properly. People who have diabetes are four times more likely to have carotid artery disease than are people who dont have diabetes. Family history of atherosclerosis. People who have a family history of atherosclerosis are more likely to develop carotid artery disease. High blood pressure (Hypertension). Blood pressure is considered high if it stays at or above 140/90 mmHg over time. If you have diabetes or chronic kidney disease, high blood pressure is defined as 130/80 mmHg or higher. (The mmHg is millimeters of mercurythe units used to measure blood pressure.) Lack of 

## Modelo base

Usamos um LLM pequeno e **nao-gated** no Hugging Face (evita a burocracia de acesso aos pesos oficiais do Llama) — `Qwen/Qwen2.5-1.5B-Instruct`. Isso ainda atende ao requisito do desafio ("LLaMA, Falcon **ou outro**").

In [13]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto'
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [14]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512, padding='max_length')

tokenized_train = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
tokenized_train.set_format(type='torch')

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

In [15]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=os.path.join(ROOT, 'results', 'finetuning', 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='no',
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)
train_result = trainer.train()
train_result

Step,Training Loss
10,1.716688
20,1.605837
30,1.537414
40,1.515796
50,1.487639
60,1.383539
70,1.376924
80,1.402742
90,1.413450


TrainOutput(global_step=96, training_loss=1.4893653690814972, metrics={'train_runtime': 203.5952, 'train_samples_per_second': 7.515, 'train_steps_per_second': 0.472, 'total_flos': 6179294486200320.0, 'train_loss': 1.4893653690814972, 'epoch': 3.0})

In [16]:
import json

ADAPTER_DIR = os.path.join(ROOT, 'results', 'finetuning', 'lora_adapter')
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

metrics = {
    'base_model': BASE_MODEL,
    'train_examples': len(train_examples),
    'eval_examples': len(eval_examples),
    'train_loss_history': [
        {'step': h.get('step'), 'loss': h.get('loss')}
        for h in trainer.state.log_history if 'loss' in h
    ],
}
with open(os.path.join(ROOT, 'results', 'finetuning', 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print('Adapter e metricas salvos em results/finetuning/')

Adapter e metricas salvos em results/finetuning/


## Avaliacao qualitativa (antes/depois)

Compara a saida do modelo base vs. o modelo com o adapter LoRA para as mesmas perguntas do conjunto de avaliacao, usando o checklist deterministico de `src/finetuning/evaluate.py` (sem GPU/API, reproduzivel).

In [17]:
from src.finetuning.dataset import format_prompt_only
from src.finetuning.evaluate import evaluate_eval_set


def generate_with_model(current_model, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors='pt').to(current_model.device)
    output = current_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text[len(prompt):].strip()


pairs_finetuned = []
for ex in eval_examples[:10]:
    prompt = format_prompt_only(ex)
    generated = generate_with_model(model, prompt)
    pairs_finetuned.append((generated, ex['output']))

result = evaluate_eval_set(pairs_finetuned)
print('Score medio (modelo fine-tuned):', result['mean_score'])

Score medio (modelo fine-tuned): 1.0


## Proximos passos

Baixe a pasta `results/finetuning/lora_adapter/` do Colab para o mesmo caminho no repositorio local. `src/assistant/llm_backend.get_generate_fn()` detecta o adapter automaticamente e passa a usar o modelo fine-tuned em vez do fallback Gemini nos notebooks 05 e 06.